# ボディ検査機 測定値トレンドレポート

指定した号機・検査項目・期間の測定値を、測定順に等間隔で表示します。

In [ ]:
import sys
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Yu Gothic"
import pandas as pd
print("実行環境:",sys.executable)

In [ ]:
# ================================================
# Settings
# ================================================
DB_PATH = Path("../data/body_insp_measurement_data.db")

STD_DATETIME = "2026-11-11 17:15:00"

INSP_MACHINES = [412, 413, 416, 421, 428]
INSP_ITEMS = [
    "total_length",
    "thickness",
    "bottom_thickness",
    "depth_phi071",
    "inner_diameter",
]

In [ ]:

def get_latest_lot_id(insp_machine_no, inspection_item, std_datetime):
    """
    指定した号機・検査項目・基準日時以前の最新測定レコードから
    ロットIDを取得する。

    Returns:
        tuple: (part_no, proc_machine_no, process_date)
        None: 該当レコードがない場合
    """
    sql = """
    SELECT part_no, proc_machine_no, process_date
    FROM measurement_result
    WHERE insp_machine_no = ?
      AND inspection_item = ?
      AND measured_at <= ?
    ORDER BY measured_at DESC
    LIMIT 1
    """

    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            sql, (insp_machine_no, inspection_item, std_datetime)
        ).fetchone()

    return row


In [ ]:

def get_lot_measurements(insp_machine_no, inspection_item, lot_id):
    """
    指定した検査機番・検査項目・ロットに一致する
    測定データをDataFrameで取得する。

    Args:
        insp_machine_no: 検査機番
        inspection_item: 検査項目
        lot_id: (part_no, proc_machine_no, process_date)

    Returns:
        pd.DataFrame: 測定日時と測定値
    """
    if lot_id is None:
        raise ValueError("ロットIDが指定されていません")

    sql = """
    SELECT *
    FROM measurement_result
    WHERE insp_machine_no = ?
      AND inspection_item = ?
      AND part_no = ?
      AND proc_machine_no = ?
      AND process_date = ?
    ORDER BY measured_at ASC
    """

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(
            sql,
            conn,
            params=(
                insp_machine_no,
                inspection_item,
                *lot_id,
            ),
        )

    return df


In [ ]:
def plot_measurement_trend(df, ax):
    """
    指定したAxesに測定値の折れ線グラフを描画する。

    Args:
        df: 測定データ（DataFrame）
        ax: 描画先のAxes
    """
    if df.empty:
        ax.set_title("No data")
        return

    insp_machine_no = df["insp_machine_no"].iloc[0]

    ax.plot(
        range(len(df)),
        df["measured_value"],
        linewidth=1,
    )

    ax.set_title(f"Machine {insp_machine_no}", loc="left")
    ax.set_xlabel("Measurement order")
    ax.set_ylabel("Measured value")
    ax.grid(True, alpha=0.3)

In [ ]:

def create_measurement_trend_figure(inspection_item, std_datetime):
    """
    指定した検査項目について、5台分の測定トレンドを
    3行×2列に配置したFigureを作成する。

    Returns:
        fig: MatplotlibのFigure
    """
    # 3行×2列のグラフ領域を作成
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))

    # 5台分のグラフを描画
    for insp_machine_no, ax in zip(INSP_MACHINES, axes.flat):

        # 基準日時からロットを取得
        lot_id = get_latest_lot_id(
            insp_machine_no,
            inspection_item,
            std_datetime,
        )

        # ロットが存在しない場合
        if lot_id is None:
            ax.set_title(f"Machine {insp_machine_no}", loc="left")
            ax.text(
                0.5, 0.5, "No data",
                ha="center", va="center",
                transform=ax.transAxes,
            )
            ax.grid(True, alpha=0.3)
            continue

        # ロットの測定データを取得
        df = get_lot_measurements(
            insp_machine_no,
            inspection_item,
            lot_id,
        )

        # 1台分のグラフを描画
        plot_measurement_trend(df, ax)

    # 余った6番目のAxesを非表示
    axes.flat[5].set_visible(False)

    fig.suptitle(f"Measurement Trend - {inspection_item}")
    fig.tight_layout()

    return fig


In [ ]:
# 検査項目ごとにFigureを作成・表示
for inspection_item in INSP_ITEMS:

    fig = create_measurement_trend_figure(
        inspection_item,
        STD_DATETIME,
    )

    plt.show()
